# DIPCOMA — Independent Testing: Variance- and Entropy-Based Ranking

**One-run Kaggle notebook.** This notebook adapts the uploaded training ranking notebook for the independent `dipcoma-test` dataset.

## Protocol
- Fit/freeze the two ranking methods using only `dipcoma-trainval` training-region patches.
- Apply the frozen band rankings to the six unseen test regions in `dipcoma-test`.
- Do not recompute ranking scores or choose bands using test data.
- Evaluate variance-ranking and entropy-ranking selections at identical Top-K budgets.
- Report region-wise and pooled test metrics. No labels are inferred from region names.

Attach both Kaggle datasets using **Add Input**:
- `krrishshandilya/dipcoma-trainval`
- `krrishshandilya/dipcoma-test`

Run all cells from top to bottom.

## 1. Configuration

`MAX_*_PATCHES_PER_REGION=None` processes all discovered patches. The default cap of 20 keeps a first run manageable; increase it for the final experiment. `AXIS_ORDER=None` detects the spectral axis as the smallest cube dimension (expected 13 bands); set it explicitly if the dataset format differs.

In [ ]:
import os, re, json, time, math, warnings, platform, zipfile, hashlib
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
TOP_K = [3, 4, 5, 6, 8, 10, 12]
ENTROPY_BINS = 256
MAX_TRAIN_PATCHES_PER_REGION = 20
MAX_TEST_PATCHES_PER_REGION = 20
AXIS_ORDER = None                 # e.g. "BHW", "HWB"; None = smallest axis is spectral
EXPECTED_BANDS = 13
RECON_TRAIN_PIXELS_PER_PATCH = 5000  # sampled training pixels used to fit reconstruction regressors
MAX_RECON_TRAIN_PIXELS = 150000    # cap per fitted regressor for memory/runtime control
RUN_RECONSTRUCTION = True         # train-fitted regression; no wavelength metadata required
OUTPUT = Path("/kaggle/working/testing_ranking_outputs") if Path("/kaggle/working").exists() else Path("testing_ranking_outputs")
for d in ["tables","rankings","selected_bands","figures","reports","reduced_cubes","config"]:
    (OUTPUT/d).mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)

## 2. Locate the two attached datasets

The notebook searches Kaggle's mounted input folders by dataset slug/name. If automatic discovery fails, set `TRAINVAL_ROOT` and `TEST_ROOT` explicitly.

In [ ]:
def find_mount(token):
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    token = token.lower().replace("-", "").replace("_", "")
    candidates = [p for p in root.iterdir() if p.is_dir()]
    for p in candidates:
        norm = p.name.lower().replace("-", "").replace("_", "")
        if token in norm:
            return p
    # search nested folders if Kaggle mount names are not descriptive
    for p in root.rglob("*"):
        if p.is_dir():
            norm = p.name.lower().replace("-", "").replace("_", "")
            if token in norm:
                return p
    return None

TRAINVAL_ROOT = find_mount("dipcoma-trainval")
TEST_ROOT = find_mount("dipcoma-test")
print("trainval:", TRAINVAL_ROOT)
print("test:", TEST_ROOT)
if TRAINVAL_ROOT is None or TEST_ROOT is None:
    raise FileNotFoundError(
        "Attach BOTH dipcoma-trainval and dipcoma-test with Kaggle Add Input, "
        "or set TRAINVAL_ROOT and TEST_ROOT to their mounted paths."
    )

## 3. Read and discover image patches

Supported formats: `.npy`, `.npz`, `.mat`, `.h5/.hdf5`, `.tif/.tiff`. Only 3-D numeric arrays with the expected spectral dimension are treated as cubes. Each file is processed independently; test regions are never used to fit rankings.

In [ ]:
def load_numeric_array(path):
    path = Path(path)
    ext = path.suffix.lower()
    if ext == ".npy":
        return np.load(path, mmap_mode="r")
    if ext == ".npz":
        z = np.load(path, allow_pickle=False)
        keys = [k for k in z.files if np.asarray(z[k]).ndim == 3]
        if not keys: return None
        return z[max(keys, key=lambda k: np.asarray(z[k]).size)]
    if ext == ".mat":
        import scipy.io
        try:
            d = scipy.io.loadmat(path)
            arrs = [v for k,v in d.items() if not k.startswith("__") and isinstance(v,np.ndarray) and v.ndim==3]
            return max(arrs, key=lambda a:a.size) if arrs else None
        except NotImplementedError:
            import h5py
            with h5py.File(path,"r") as f:
                arrs=[]
                f.visititems(lambda n,o: arrs.append(o[()]) if isinstance(o,h5py.Dataset) and len(o.shape)==3 else None)
            return max(arrs,key=lambda a:a.size) if arrs else None
    if ext in (".h5",".hdf5"):
        import h5py
        with h5py.File(path,"r") as f:
            arrs=[]
            f.visititems(lambda n,o: arrs.append(o[()]) if isinstance(o,h5py.Dataset) and len(o.shape)==3 else None)
        return max(arrs,key=lambda a:a.size) if arrs else None
    if ext in (".tif",".tiff"):
        try:
            import rasterio
            with rasterio.open(path) as ds: return ds.read()
        except Exception:
            import tifffile
            return tifffile.imread(path)
    return None

def to_bhw(arr):
    a=np.asarray(arr)
    if a.ndim != 3: return None
    if AXIS_ORDER:
        if sorted(AXIS_ORDER) != sorted("BHW"): raise ValueError("AXIS_ORDER must be BHW, BHW permutation such as HWB/HBW")
        return np.transpose(a,[AXIS_ORDER.index(c) for c in "BHW"])
    # Prefer an axis exactly equal to EXPECTED_BANDS; otherwise use the smallest dimension.
    matches=[i for i,n in enumerate(a.shape) if n==EXPECTED_BANDS]
    axis=matches[0] if matches else int(np.argmin(a.shape))
    return np.moveaxis(a,axis,0)

VALID_EXTS={".npy",".npz",".mat",".h5",".hdf5",".tif",".tiff"}
def discover_cubes(root, split=None):
    root=Path(root)
    files=[]
    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in VALID_EXTS: continue
        relparts=[x.lower() for x in p.relative_to(root).parts]
        if split and split not in relparts: continue
        # Skip obvious labels/masks and metadata
        if re.search(r"(label|mask|ground.?truth|class|annotation|metadata)",p.stem,re.I): continue
        try:
            arr=load_numeric_array(p)
            cube=to_bhw(arr) if arr is not None else None
            if cube is not None and cube.shape[0] >= 3 and cube.shape[1] > 8 and cube.shape[2] > 8:
                files.append(p)
        except Exception as e:
            pass
    return sorted(files)

train_files=discover_cubes(TRAINVAL_ROOT, split="train")
test_files=discover_cubes(TEST_ROOT)
print("Discovered train cubes:",len(train_files))
print("Discovered test cubes:",len(test_files))
print("Example train:",[str(p.relative_to(TRAINVAL_ROOT)) for p in train_files[:5]])
print("Example test:",[str(p.relative_to(TEST_ROOT)) for p in test_files[:5]])
if not train_files or not test_files:
    raise RuntimeError("No 3-D cubes found. Inspect dataset structure and set AXIS_ORDER if required.")

## 4. Region grouping and deterministic sampling

The region is taken from the first directory below the dataset root (or the split directory for trainval). Patches are sampled deterministically within each region. The notebook prints the resulting counts so the sampling protocol is auditable.

In [ ]:
def region_name(path, root, train=False):
    parts=list(Path(path).relative_to(root).parts[:-1])
    # Remove generic container/split folders wherever they occur.
    ignored={"dataset","datasets","data","train","training","val","valid","validation","test"}
    meaningful=[p for p in parts if p.lower() not in ignored]
    return meaningful[0] if meaningful else "unknown_region"

def sample_by_region(files, root, cap, train=False):
    groups=defaultdict(list)
    for p in files: groups[region_name(p,root,train)].append(p)
    rng=np.random.default_rng(SEED)
    chosen={}
    for reg, paths in sorted(groups.items()):
        paths=sorted(paths)
        if cap is not None and len(paths)>cap:
            idx=sorted(rng.choice(len(paths),cap,replace=False).tolist())
            paths=[paths[i] for i in idx]
        chosen[reg]=paths
    return chosen

train_groups=sample_by_region(train_files,TRAINVAL_ROOT,MAX_TRAIN_PATCHES_PER_REGION,train=True)
test_groups=sample_by_region(test_files,TEST_ROOT,MAX_TEST_PATCHES_PER_REGION,train=False)
print("TRAIN PATCHES BY REGION:",{k:len(v) for k,v in train_groups.items()})
print("TEST PATCHES BY REGION:",{k:len(v) for k,v in test_groups.items()})

## 5. Fit/freeze ranking scores on training regions only

Variance and Shannon entropy are calculated per spectral band on training patches. Scores are first computed per patch and then averaged across training patches, giving each patch equal weight. The ranking is global across training regions. Test arrays are not accessed in this section.

In [ ]:
def cube_stats(path):
    arr=load_numeric_array(path)
    cube=to_bhw(arr).astype(np.float32,copy=False)
    if cube.shape[0] != EXPECTED_BANDS:
        raise ValueError(f"{path}: expected {EXPECTED_BANDS} bands, got {cube.shape}")
    mask=np.isfinite(cube).all(axis=0)
    if not mask.any(): raise ValueError(f"No valid pixels in {path}")
    x=cube[:,mask].astype(np.float64)
    # variance score on original numeric values
    var=np.var(x,axis=1,ddof=0)
    # common per-patch range across all bands; same bins for all bands within this patch
    lo=float(np.min(x)); hi=float(np.max(x))
    if hi<=lo:
        ent=np.zeros(cube.shape[0],dtype=float)
    else:
        edges=np.linspace(lo,hi,ENTROPY_BINS+1)
        ent=np.array([-(np.histogram(row,bins=edges)[0]/row.size *
                         np.log2(np.maximum(np.histogram(row,bins=edges)[0]/row.size,1e-300))).sum()
                      for row in x],dtype=float)
    return var,ent,cube,mask

train_records=[]
for region,paths in train_groups.items():
    for path in paths:
        try:
            v,e,_,_=cube_stats(path)
            train_records.append((region,str(path),v,e))
        except Exception as exc:
            print("SKIP TRAIN",path,repr(exc))
if not train_records: raise RuntimeError("No usable training cubes.")
BANDS=EXPECTED_BANDS
VAR_TRAIN=np.mean([r[2] for r in train_records],axis=0)
ENT_TRAIN=np.mean([r[3] for r in train_records],axis=0)
var_order=np.lexsort((np.arange(BANDS),-VAR_TRAIN))
ent_order=np.lexsort((np.arange(BANDS),-ENT_TRAIN))
print("Training patches used:",len(train_records))
print("Variance ranking (0-based):",var_order.tolist())
print("Entropy ranking (0-based):",ent_order.tolist())
pd.DataFrame({"band_0based":np.arange(BANDS),"variance_train":VAR_TRAIN,"variance_rank":np.argsort(np.argsort(-VAR_TRAIN))+1,
              "entropy_train":ENT_TRAIN,"entropy_rank":np.argsort(np.argsort(-ENT_TRAIN))+1}).to_csv(OUTPUT/"rankings"/"frozen_training_rankings.csv",index=False)

## 6. Freeze Top-K selections

The selected band indices are derived only from training scores. The same selections are reused unchanged for every test region and patch.

In [ ]:
K_LIST=sorted(set(int(k) for k in TOP_K if 1<=int(k)<=BANDS))
if not K_LIST: raise ValueError("TOP_K contains no valid values.")
if max(K_LIST)>BANDS: raise ValueError(f"Top-K cannot exceed {BANDS} bands.")
SELECTIONS={
    "Variance":{k:var_order[:k].copy() for k in K_LIST},
    "Entropy":{k:ent_order[:k].copy() for k in K_LIST}
}
for method in SELECTIONS:
    for k,idx in SELECTIONS[method].items():
        pd.DataFrame({"rank":np.arange(1,k+1),"band_0based":idx,"band_1based":idx+1}).to_csv(
            OUTPUT/"selected_bands"/f"{method.lower()}_top{k}.csv",index=False)
print({m:{k:v.tolist() for k,v in d.items()} for m,d in SELECTIONS.items()})

In [ ]:
# Fit one multivariate linear reconstruction model for each frozen ranking method and K.
# Crucially, only dipcoma-trainval/train patches are used here; test patches are never used to fit.
from sklearn.linear_model import LinearRegression

# Estimate a single fixed radiometric range from training patches only.
TRAIN_VALUE_MIN = np.inf
TRAIN_VALUE_MAX = -np.inf
for _region, _paths in sorted(train_groups.items()):
    for _path in _paths:
        try:
            _arr = load_numeric_array(_path)
            _cube = to_bhw(_arr).astype(np.float32, copy=False)
            _valid = np.isfinite(_cube).all(axis=0)
            if _valid.any():
                _vals = _cube[:, _valid]
                TRAIN_VALUE_MIN = min(TRAIN_VALUE_MIN, float(np.min(_vals)))
                TRAIN_VALUE_MAX = max(TRAIN_VALUE_MAX, float(np.max(_vals)))
        except Exception as _exc:
            print("SKIP TRAIN RANGE", _path, repr(_exc))
if not np.isfinite(TRAIN_VALUE_MIN) or not np.isfinite(TRAIN_VALUE_MAX):
    raise RuntimeError("Could not determine training-only radiometric range.")
TRAIN_DYNAMIC_RANGE = max(TRAIN_VALUE_MAX - TRAIN_VALUE_MIN, 1.0)

def fit_reconstruction_model(selected, seed=SEED):
    selected = np.asarray(selected, dtype=int)
    omitted = np.setdiff1d(np.arange(BANDS), selected)
    if len(omitted) == 0:
        return {"selected": selected, "omitted": omitted, "model": None,
                "train_pixels": 0, "train_fit_time_s": 0.0}
    rng = np.random.default_rng(seed)
    X_parts, Y_parts = [], []
    remaining = int(MAX_RECON_TRAIN_PIXELS)
    for region, paths in sorted(train_groups.items()):
        for path in paths:
            if remaining <= 0:
                break
            try:
                _, _, cube, mask = cube_stats(path)
                flat = cube.reshape(BANDS, -1).astype(np.float32, copy=False)
                valid_ids = np.flatnonzero(mask.reshape(-1))
                if not len(valid_ids):
                    continue
                n = min(RECON_TRAIN_PIXELS_PER_PATCH, len(valid_ids), remaining)
                ids = rng.choice(valid_ids, size=n, replace=False) if len(valid_ids) > n else valid_ids[:n]
                X_parts.append(flat[selected][:, ids].T)
                Y_parts.append(flat[omitted][:, ids].T)
                remaining -= len(ids)
            except Exception as exc:
                print("SKIP RECON TRAIN", path, repr(exc))
        if remaining <= 0:
            break
    if not X_parts:
        raise RuntimeError("No training pixels available for fitting reconstruction regression.")
    X_train = np.concatenate(X_parts, axis=0)
    Y_train = np.concatenate(Y_parts, axis=0)
    t0 = time.perf_counter()
    model = LinearRegression()
    model.fit(X_train, Y_train)
    elapsed = time.perf_counter() - t0
    return {"selected": selected, "omitted": omitted, "model": model,
            "train_pixels": int(len(X_train)), "train_fit_time_s": float(elapsed)}

RECON_MODELS = {}
recon_fit_rows = []
for method, by_k in SELECTIONS.items():
    for k, selected in by_k.items():
        key = (method, int(k))
        fitted = fit_reconstruction_model(selected, seed=SEED + int(k))
        RECON_MODELS[key] = fitted
        recon_fit_rows.append({
            "method": method, "K": int(k),
            "selected_bands_0based": json.dumps(np.asarray(selected, dtype=int).tolist()),
            "omitted_bands_0based": json.dumps(fitted["omitted"].tolist()),
            "training_pixels": fitted["train_pixels"],
            "regression_fit_time_s": fitted["train_fit_time_s"]
        })
pd.DataFrame(recon_fit_rows).to_csv(OUTPUT/"tables"/"reconstruction_regression_training.csv", index=False)
print("Fitted train-only reconstruction regressors:", len(RECON_MODELS))

## 7. Test metrics and reconstruction

For each frozen ranking selection, a multivariate linear regression model is fitted on sampled **training** pixels to predict omitted bands from selected bands. The fitted regressor is then applied unchanged to each independent test patch.

Reported metrics:
- PSNR using a dynamic range calculated from training data only;
- SSIM on reconstructed omitted bands;
- SAM across full pixel spectra;
- MAE and MRE;
- selected-band redundancy;
- theoretical selected-band payload ratio and bpp estimate.

The test dataset is never used to fit regression coefficients, calculate the training dynamic range, rank bands, or select K. These are not measured JPEG2000 bitstream compression rates.

In [ ]:
from skimage.metrics import structural_similarity

def reconstruct_from_selected(cube, selected, recon_model):
    selected = np.asarray(selected, dtype=int)
    omitted = np.setdiff1d(np.arange(BANDS), selected)
    recon = np.asarray(cube, dtype=np.float32).copy()
    if len(omitted):
        if recon_model is None:
            raise RuntimeError("Missing train-fitted reconstruction model.")
        flat = np.asarray(cube, dtype=np.float32).reshape(BANDS, -1)
        # Predict in chunks to avoid a large temporary matrix on 512x512 patches.
        predicted = np.empty((len(omitted), flat.shape[1]), dtype=np.float32)
        chunk = 100000
        for start in range(0, flat.shape[1], chunk):
            end = min(start + chunk, flat.shape[1])
            x_chunk = flat[selected, start:end].T
            predicted[:, start:end] = recon_model.predict(x_chunk).T
        recon.reshape(BANDS, -1)[omitted] = predicted
    # Preserve selected bands exactly; only omitted bands are reconstructed.
    recon[selected] = np.asarray(cube, dtype=np.float32)[selected]
    return recon

def test_patch_metrics(path, method, k, selected):
    arr = load_numeric_array(path)
    cube = to_bhw(arr).astype(np.float32, copy=False)
    if cube.shape[0] != BANDS:
        raise ValueError(f"Band count mismatch in {path}: {cube.shape[0]} vs {BANDS}")
    mask = np.isfinite(cube).all(axis=0)
    if not mask.any():
        raise ValueError(f"No valid pixels in {path}")
    # Invalid pixels are excluded from metric calculations.
    x = cube.astype(np.float64)
    selected = np.asarray(selected, dtype=int)
    key = (method, int(k))
    fitted = RECON_MODELS[key]
    recon = reconstruct_from_selected(cube, selected, fitted["model"]).astype(np.float64)
    valid = mask
    xv = x[:, valid]
    yv = recon[:, valid]
    err = xv - yv
    mse = float(np.mean(err ** 2))
    mae = float(np.mean(np.abs(err)))
    psnr = float("inf") if mse <= 0 else float(10.0 * np.log10((TRAIN_DYNAMIC_RANGE ** 2) / mse))

    # Spectral Angle Mapper, averaged over valid pixels.
    dot = np.sum(xv * yv, axis=0)
    denom = np.linalg.norm(xv, axis=0) * np.linalg.norm(yv, axis=0)
    ok = denom > 1e-12
    sam = float(np.degrees(np.arccos(np.clip(dot[ok] / denom[ok], -1.0, 1.0))).mean()) if ok.any() else np.nan

    # SSIM averaged over omitted bands, evaluated on a downsampled image for speed.
    omitted = np.setdiff1d(np.arange(BANDS), selected)
    ssim_vals = []
    step = 4
    for b in omitted:
        a = x[b, ::step, ::step]
        z = recon[b, ::step, ::step]
        data_range = float(max(TRAIN_DYNAMIC_RANGE, np.max(a) - np.min(a), 1.0))
        ssim_vals.append(float(structural_similarity(a, z, data_range=data_range, win_size=7)))
    ssim = float(np.mean(ssim_vals)) if ssim_vals else 1.0

    # Relative error only where the reference magnitude is not near zero.
    rel_eps = 1e-4 * TRAIN_DYNAMIC_RANGE
    mre_vals = []
    for b in omitted:
        band_ref = x[b, valid]
        band_err = np.abs(err[b])
        keep = np.abs(band_ref) > rel_eps
        if keep.any():
            mre_vals.append(float(np.mean(band_err[keep] / np.abs(band_ref[keep]))))
    mre = float(np.mean(mre_vals)) if mre_vals else np.nan

    selected_cube = cube[selected][:, valid].astype(np.float64)
    if len(selected) > 1:
        corr = np.corrcoef(selected_cube)
        tri = np.triu_indices(len(selected), 1)
        redundancy = float(np.nanmean(np.abs(corr[tri])))
    else:
        redundancy = np.nan

    bits = np.dtype(arr.dtype).itemsize * 8
    return {
        "method": method, "K": int(k), "region": region_name(path, TEST_ROOT),
        "patch": Path(path).name,
        "selected_0based": json.dumps(selected.tolist()),
        "n_valid_pixels": int(valid.sum()),
        "CR_theoretical": float(BANDS / len(selected)),
        "bpp_per_original_spectral_sample": float(len(selected) / BANDS),
        "bits_per_spatial_pixel": int(len(selected) * bits),
        "selected_redundancy": redundancy,
        "PSNR_dB": psnr, "SSIM": ssim, "SAM_deg": sam, "MAE": mae, "MRE": mre,
        "regression_fit_time_s": float(fitted["train_fit_time_s"]),
        "original_dtype": str(arr.dtype), "assumed_bits_per_sample": bits
    }

test_rows = []
errors = []
for region, paths in test_groups.items():
    for path in paths:
        for method in SELECTIONS:
            for k, selected in SELECTIONS[method].items():
                try:
                    test_rows.append(test_patch_metrics(path, method, k, selected))
                except Exception as exc:
                    errors.append({"path": str(path), "method": method, "K": k, "error": repr(exc)})
                    print("SKIP TEST", path, method, k, repr(exc))
TEST_RESULTS = pd.DataFrame(test_rows)
if TEST_RESULTS.empty:
    raise RuntimeError("No test results were produced; inspect test cube format and errors.")
TEST_RESULTS.to_csv(OUTPUT / "tables" / "test_patch_metrics.csv", index=False)
pd.DataFrame(errors).to_csv(OUTPUT / "reports" / "skipped_test_patches.csv", index=False)
print("Successful patch-method-K evaluations:", len(TEST_RESULTS))
print("Skipped evaluations:", len(errors))
print("Reconstruction metrics computed using train-fitted linear regression.")

## 8. Region-wise and pooled test comparison

The pooled summary is the mean across patch-level rows. Region-wise summaries are also reported so a strong result in one region cannot hide poor performance elsewhere. Since CR and bpp are determined by K, they are theoretical storage ratios and will be identical for both ranking methods at the same K.

In [ ]:
metric_cols=["selected_redundancy","PSNR_dB","SSIM","SAM_deg","MAE","MRE","CR_theoretical","bpp_per_original_spectral_sample"]
region_summary=TEST_RESULTS.groupby(["region","method","K"],dropna=False)[metric_cols].mean(numeric_only=True).reset_index()
pooled_summary=TEST_RESULTS.groupby(["method","K"],dropna=False)[metric_cols].agg(["mean","std","count"])
region_summary.to_csv(OUTPUT/"tables"/"region_wise_test_summary.csv",index=False)
pooled_summary.to_csv(OUTPUT/"tables"/"pooled_test_summary.csv")
print("POOLED TEST SUMMARY")
display(pooled_summary)
print("REGION-WISE TEST SUMMARY")
display(region_summary)

## 9. Visualizations

These figures are generated from independent test results. The training rankings remain frozen.

In [ ]:
plot_metrics = [
    ("PSNR_dB", "PSNR (dB) — higher is better"),
    ("SSIM", "SSIM — higher is better"),
    ("SAM_deg", "SAM (degrees) — lower is better"),
    ("MAE", "MAE — lower is better"),
    ("selected_redundancy", "Selected-band redundancy — lower is better")
]
for metric, title in plot_metrics:
    fig, ax = plt.subplots(figsize=(8, 4))
    for method in SELECTIONS:
        d = pooled_summary.loc[method]
        if (metric, "mean") in d.columns:
            ax.plot(d.index, d[(metric, "mean")], marker="o", label=method)
    ax.set_xlabel("Retained bands (K)")
    ax.set_ylabel(metric)
    ax.set_title(title)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No finite values available", ha="center", va="center", transform=ax.transAxes)
    fig.tight_layout()
    fig.savefig(OUTPUT / "figures" / f"{metric}.png", dpi=160)
    plt.show()

## 10. Protocol checks and final report

The checks below record the source roots, selected bands, number of training/test patches and the fact that test data was not used to fit rankings.

In [ ]:
summary={
 "protocol":"rankings fitted on train split only; frozen selections evaluated on independent test regions",
 "trainval_root":str(TRAINVAL_ROOT),"test_root":str(TEST_ROOT),
 "training_patches_used":len(train_records),
 "training_regions":{k:len(v) for k,v in train_groups.items()},
 "test_regions":{k:len(v) for k,v in test_groups.items()},
 "test_patch_count_successful":int(TEST_RESULTS[["region","patch"]].drop_duplicates().shape[0]),
 "test_evaluations":int(len(TEST_RESULTS)),
 "skipped_evaluations":int(len(errors)),
 "bands":BANDS,"top_k":K_LIST,"entropy_bins":ENTROPY_BINS,
 "frozen_selections":{m:{str(k):v.tolist() for k,v in d.items()} for m,d in SELECTIONS.items()},
 "reconstruction_method":"multivariate linear regression fitted on trainval/train only",
 "training_dynamic_range":float(TRAIN_DYNAMIC_RANGE),
 "reconstruction_metrics_available":True,
 "seed":SEED,"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__
}
(OUTPUT/"reports"/"run_summary.json").write_text(json.dumps(summary,indent=2))
(OUTPUT/"reports"/"README.txt").write_text(
 "DIPCOMA independent testing results for variance- and entropy-based band ranking.\n"
 "Rankings were fitted on dipcoma-trainval training patches only. Selected bands were frozen before evaluating dipcoma-test.\n"
 "CR/bpp are theoretical selected-band storage measures, not measured codec bitstreams.\n"
 "Reconstruction metrics are N/A unless real wavelength coordinates are supplied.\n"
)
zip_path=OUTPUT.parent/"testing_ranking_outputs.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for f in OUTPUT.rglob("*"):
        if f.is_file(): z.write(f,f.relative_to(OUTPUT.parent))
print(json.dumps(summary,indent=2))
print("Results folder:",OUTPUT.resolve())
print("ZIP:",zip_path.resolve())